In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)


In [ ]:
# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df

# 1. Encoding categorical **data**

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe


In [ ]:
# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

In [ ]:

# Example size order, if not already defined:
size_order = ['S', 'M', 'L', 'XL']

# 1. One-hot encode just the 'city' column of the whole DataFrame
df_encoded = pd.get_dummies(df, columns=['city'])

# 2. Ordinal-encode 'size' using size_order
size_code = {size: i for i, size in enumerate(size_order)}
df_encoded['size'] = df_encoded['size'].map(size_code)

print(df_encoded)

# 3. Why one-hot is wrong for size / ordinal is wrong for city:
# One-hot is wrong for size because size has a natural order (S < M < L < XL).
# Ordinal encoding is wrong for city because cities have no natural ranking or order.

# 2. Feature **scaling**

In [ ]:

# -----------------------------------------------------------
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# -----------------------------------------------------------

# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])


In [ ]:
# -----------------------------------------------------------
# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

income = df[['income']]   # 2D shape for sklearn

# 1. Standardise -> check mean ~0, std ~1
standard_scaler = StandardScaler()
income_standard = standard_scaler.fit_transform(income)

print("Standardised mean:", income_standard.mean())
print("Standardised std:", income_standard.std())

# 2. Min-Max scale -> check min 0, max 1
minmax_scaler = MinMaxScaler()
income_minmax = minmax_scaler.fit_transform(income)

print("Min-Max minimum:", income_minmax.min())
print("Min-Max maximum:", income_minmax.max())

# 3. Needs scaling: K-Nearest Neighbors (KNN)
#    Doesn't need scaling: Decision Tree

# **3. Avoiding data leakage**

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))



In [ ]:
from sklearn.preprocessing import StandardScaler

# 1a. WRONG: fit on ALL data, then split (leakage!)
wrong_mean = StandardScaler().fit(X).mean_

# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem:
# Fitting on all data allows information from the test set to influence
# the preprocessing parameters (such as the mean and standard deviation).
# This causes data leakage and can make test performance look better than
# it really is. The scaler should be fitted only on the training data.

# **4. Feature engineering**

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS
# -----------------------------------------------------------

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. EXTRACT FROM A DATE
# -----------------------------------------------------------

dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d


In [ ]:
import pandas as pd

ex = df.copy()

# 1. high_earner flag (income > median)
median_income = ex['income'].median()
ex['high_earner'] = ex['income'] > median_income

# 2. Bin income into 3 buckets with pd.cut
ex['income_bucket'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['low', 'mid', 'high']
)

# 3. Show income + the new columns
print(ex[['income', 'high_earner', 'income_bucket']])

# **5. The preprocessing pipeline**

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

# scale the numbers, one-hot the categories — all in one object
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)


In [ ]:
# -----------------------------------------------------------
# 🔹 5B. FIT THE WHOLE THING IN ONE CALL
# -----------------------------------------------------------

X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)        # preprocessing fitted on TRAIN only — no leakage
acc = pipe.score(X_test, y_test)  # transforms test with train-fitted steps
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

Xn = df[['age', 'income']]
yn = df['bought']

# 1. Pipeline: MinMaxScaler -> LogisticRegression
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression())
])

# 2. Split + fit on training data
X_train, X_test, y_train, y_test = train_test_split(
    Xn, yn,
    test_size=0.2,
    random_state=42
)

pipe.fit(X_train, y_train)

# 3. Print test accuracy
print("Test accuracy:", pipe.score(X_test, y_test))